In [0]:
from pyspark.sql import functions as F

base_path = "dbfs:/Volumes/workspace/dinedash/raw_data"

dimensions_path = f"{base_path}/dimensions"
transactions_path = f"{base_path}/transactions"

df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_customers.csv")
)

df_restaurants = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_restaurants.csv")
)

df_delivery_agents = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_delivery_agents.csv")
)

df_locations = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_locations.csv")
)

df_menu_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{dimensions_path}/dim_menu_items.csv")
)

df_orders = (
    spark.read
    .option("multiline", "true")
    .json(f"{base_path}/orders/*.json")
)

In [0]:
print("Customers:", df_customers.count())
print("Restaurants:", df_restaurants.count())
print("Delivery Agents:", df_delivery_agents.count())
print("Locations:", df_locations.count())
print("Menu Items:", df_menu_items.count())
print("Orders:", df_orders.count())

In [0]:
df_orders.printSchema()
display(df_orders.limit(5))

In [0]:
from pyspark.sql.functions import explode, to_timestamp

df_orders_clean = (
    df_orders
    .withColumn("order_timestamp", to_timestamp("timestamp"))
    .withColumn("item", explode("items_ordered"))
    .select(
        "order_id",
        "customer_id",
        "restaurant_id",
        "agent_id",
        "delivery_location_id",
        "order_status",
        "payment_method",
        "order_timestamp",
        "tip",
        "total_amount",
        "item.item_id",
        "item.item_name",
        "item.price",
        "item.quantity"
    )
)

In [0]:
display(df_orders_clean.limit(5))

In [0]:
display(df_customers.limit(5))

In [0]:
from pyspark.sql.functions import col

df_orders_clean = df_orders.select(
    col("order_id"),
    col("customer_id"),
    col("restaurant_id"),
    col("agent_id"),
    col("delivery_location_id"),
    col("order_status"),
    col("payment_method"),
    col("timestamp").alias("order_timestamp"),
    col("tip"),
    col("total_amount"),
    col("items_ordered")
)

In [0]:
df_orders_clean.printSchema()

display(df_orders_clean.limit(5))

In [0]:
from pyspark.sql.functions import explode, col

df_order_items = (
    df_orders_clean
    .select(
        col("order_id"),
        explode("items_ordered").alias("item")
    )
    .select(
        col("order_id"),
        col("item.item_id").alias("item_id"),
        col("item.item_name").alias("item_name"),
        col("item.price").alias("item_price"),
        col("item.quantity").alias("quantity")
    )
)

In [0]:
df_order_items.printSchema()

display(df_order_items.limit(10))

In [0]:
df_customers.createOrReplaceTempView("customers_raw")

df_restaurants.createOrReplaceTempView("restaurants_raw")

df_delivery_agents.createOrReplaceTempView("delivery_agents_raw")

df_locations.createOrReplaceTempView("locations_raw")

df_menu_items.createOrReplaceTempView("menu_items_raw")

df_orders.createOrReplaceTempView("orders_raw")

In [0]:
%sql
SELECT *
FROM customers_raw
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(customer_id) AS null_customer_ids
FROM customers_raw;

In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS count
FROM customers_raw
GROUP BY customer_id
HAVING COUNT(*) > 1;

In [0]:
df_customers.createOrReplaceTempView("customers")

In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS count
FROM customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY count DESC;

In [0]:
%sql
SELECT *
FROM customers
WHERE customer_id = 'C1004';

In [0]:
df_customers_clean = df_customers.dropDuplicates()

In [0]:
display(df_customers_clean.limit(10))

In [0]:
from pyspark.sql.functions import col, sum

df_customers_clean.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_customers_clean.columns
    ]
).show()

In [0]:
df_order_items.printSchema()

display(df_order_items.limit(10))

In [0]:
df_order_items.createOrReplaceTempView("order_items")

In [0]:
%sql

SELECT
    order_id,
    item_id,
    COUNT(*) AS count
FROM order_items
GROUP BY
    order_id,
    item_id
HAVING COUNT(*) > 1
ORDER BY count DESC;

In [0]:
%sql

SELECT *
FROM order_items
WHERE item_price <= 0
   OR quantity <= 0;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS null_item_id,
    SUM(CASE WHEN item_name IS NULL THEN 1 ELSE 0 END) AS null_item_name,
    SUM(CASE WHEN item_price IS NULL THEN 1 ELSE 0 END) AS null_item_price,
    SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS null_quantity
FROM order_items;

In [0]:
%sql

SELECT
    MIN(item_price) AS min_price,
    MAX(item_price) AS max_price,
    AVG(item_price) AS avg_price,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN item_price = 0 THEN 1 ELSE 0 END) AS zero_price_rows,
    SUM(CASE WHEN item_price < 0 THEN 1 ELSE 0 END) AS negative_price_rows
FROM order_items;

In [0]:
%sql

SELECT
    item_id,
    item_name,
    COUNT(*) AS zero_price_count
FROM order_items
WHERE item_price = 0
GROUP BY
    item_id,
    item_name
ORDER BY zero_price_count DESC;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
print([x for x in dir() if x.startswith("df_") or x.startswith("clean")])

In [0]:
tables = {
    "customers": df_customers,
    "restaurants": df_restaurants,
    "delivery_agents": df_delivery_agents,
    "locations": df_locations,
    "menu_items": df_menu_items,
    "orders": df_orders,
    "order_items": df_order_items
}

for name, df in tables.items():
    if hasattr(df, "createOrReplaceTempView"):
        df.createOrReplaceTempView(f"raw_{name}")
    else:
        spark.createDataFrame(df).createOrReplaceTempView(f"raw_{name}")

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.customers AS
SELECT * FROM raw_customers;

CREATE OR REPLACE TABLE bronze.restaurants AS
SELECT * FROM raw_restaurants;

CREATE OR REPLACE TABLE bronze.delivery_agents AS
SELECT * FROM raw_delivery_agents;

CREATE OR REPLACE TABLE bronze.locations AS
SELECT * FROM raw_locations;

CREATE OR REPLACE TABLE bronze.menu_items AS
SELECT * FROM raw_menu_items;

CREATE OR REPLACE TABLE bronze.orders AS
SELECT * FROM raw_orders;

CREATE OR REPLACE TABLE bronze.order_items AS
SELECT * FROM raw_order_items;